In [1]:
import json
import pandas as pd
import os
# print(os.getcwd())
# print(os.path.abspath(base_path))


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [2]:
# ls

In [3]:
base_path = "../results_strength_main/balanced"

folders = {

    3: "test_eval_service_3",
    5: "test_eval_service_5",
    8: "test_eval_service_8",
    10: "test_eval_service_10",
    12: "test_eval_service_12",
    14: "test_eval_service_14"

}

In [4]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 1600 rows
K=5: loaded 2400 rows
K=8: loaded 3600 rows
K=10: loaded 4400 rows
K=12: loaded 5200 rows
K=14: loaded 6000 rows


In [5]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])
df_k12 = pd.DataFrame(data[12])
df_k14 = pd.DataFrame(data[14])


print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)
print("K=12:", df_k12.shape)
print("K=14:", df_k14.shape)

K=3: (1600, 39)
K=5: (2400, 39)
K=8: (3600, 39)
K=10: (4400, 39)
K=12: (5200, 39)
K=14: (6000, 39)


In [6]:
# df_k3["model_name"].unique()
# df_k3.columns

## combined df

In [7]:
df_combined = pd.concat([df_k3, df_k5, df_k8, df_k10, df_k12, df_k14], ignore_index=True)

print(df_combined.shape)
print(df_combined["total_service_count"].value_counts())

(23200, 39)
14    6000
12    5200
10    4400
8     3600
5     2400
3     1600
Name: total_service_count, dtype: int64


In [12]:
df = df_combined[df_combined["routing_method"] == "bi-encoder"].copy()

df["overall_acc"] = (df["expected_service"] == df["predicted_label"]).astype(float)
df["specialist_recall"] = df["overall_acc"].where(df["expected_service"] != "general")
df["general_recall"]    = df["overall_acc"].where(df["expected_service"] == "general")

out = df.pivot_table(
    index=["routing_method", "model_name", "total_service_count"],
    values=["overall_acc", "specialist_recall", "general_recall"],
    aggfunc="mean",
)
out["balanced_acc"] = (out["specialist_recall"] + out["general_recall"]) / 2

out = out[["overall_acc", "specialist_recall", "general_recall", "balanced_acc"]]

pct_cols = ["specialist_recall", "general_recall", "overall_acc", "balanced_acc"]
out[pct_cols] = out[pct_cols] * 100

out.round(1)

# out.isna().sum()

overall_acc  \
routing_method model_name              total_service_count                
bi-encoder     BAAI/bge-large-en-v1.5  3                           71.2   
                                       5                           63.3   
                                       8                           69.0   
                                       10                          71.7   
                                       12                          67.3   
                                       14                          67.5   
               Qwen/Qwen3-Embedding-4B 3                           81.8   
                                       5                           59.3   
                                       8                           54.0   
                                       10                          53.8   
                                       12                          47.5   
                                       14                          46.3   

                                                            specialist_recall  \
routing_method model_name              total_service_count                      
bi-encoder     BAAI/bge-large-en-v1.5  3                                 72.7   
                                       5                                 60.0   
                                       8                                 68.0   
                                       10                                71.0   
                                       12                                67.5   
                                       14                                68.1   
               Qwen/Qwen3-Embedding-4B 3                                 86.0   
                                       5                                 62.6   
                                       8                                 50.1   
                                       10                                50.0   
                                       12                                44.3   
                                       14                                44.2   

                                                            general_recall  \
routing_method model_name              total_service_count                   
bi-encoder     BAAI/bge-large-en-v1.5  3                              67.0   
                                       5                              80.0   
                                       8                              77.0   
                                       10                             79.0   
                                       12                             65.0   
                                       14                             60.0   
               Qwen/Qwen3-Embedding-4B 3                              69.0   
                                       5                              43.0   
                                       8                              85.0   
                                       10                             92.0   
                                       12                             85.0   
                                       14                             76.0   

                                                            balanced_acc  
routing_method model_name              total_service_count                
bi-encoder     BAAI/bge-large-en-v1.5  3                            69.8  
                                       5                            70.0  
                                       8                            72.5  
                                       10                           75.0  
                                       12                           66.3  
                                       14                           64.0  
               Qwen/Qwen3-Embedding-4B 3                            77.5  
                                       5                            52.8  
                               

### inspect a model

In [36]:
MODEL = "BAAI/bge-large-en-v1.5"

In [37]:
d3 = df_k3[df_k3["model_name"] == MODEL].copy()

d5 = df_k5[df_k5["model_name"] == MODEL].copy()

d8 = df_k8[df_k8["model_name"] == MODEL].copy()

d10 = df_k10[df_k10["model_name"] == MODEL].copy()
d12 = df_k12[df_k12["model_name"] == MODEL].copy()
d14 = df_k14[df_k14["model_name"] == MODEL].copy()


print("K=3", d3.shape)
print("K=5", d5.shape)
print("K=8", d8.shape)

print("K=10", d10.shape)
print("K=12", d12.shape)
print("K=14", d14.shape)


K=3 (400, 39)
K=5 (600, 39)
K=8 (900, 39)
K=10 (1100, 39)
K=12 (1300, 39)
K=14 (1500, 39)


In [38]:
def expand_scores_and_zscores(df):
    """
    Expand the nested `scores` and `zscores` dict columns
    into normal dataframe columns.
    """

    scores_df = (
        pd.json_normalize(df["scores"])
        .add_suffix("_score")
    )

    zscores_df = (
        pd.json_normalize(df["zscores"])
        .add_suffix("_zscore")
    )

    df_expanded = pd.concat(
        [
            df.reset_index(drop=True),
            scores_df.reset_index(drop=True),
            zscores_df.reset_index(drop=True),
        ],
        axis=1,
    )

    return df_expanded

In [39]:
d3 = expand_scores_and_zscores(d3)
d5 = expand_scores_and_zscores(d5)
d8 = expand_scores_and_zscores(d8)
d10 = expand_scores_and_zscores(d10)
d12 = expand_scores_and_zscores(d12)
d14 = expand_scores_and_zscores(d14)

In [40]:
# d3.columns.to_list()

In [41]:
def recall_summary(df):

    specialist_df = df[
        df["expected_service"] != "general"
    ]

    general_df = df[
        df["expected_service"] == "general"
    ]

    accuracy = df["is_correct"].mean()

    specialist_recall = (
        specialist_df["predicted_label"]
        == specialist_df["expected_service"]
    ).mean()

    general_recall = (
        general_df["predicted_label"]
        == "general"
    ).mean()

    print(
        f"Accuracy: {accuracy:.4f}"
    )

    print(
        f"Specialist Recall: "
        f"{specialist_recall:.4f}"
    )

    print(
        f"General Recall: "
        f"{general_recall:.4f}"
    )


In [42]:
# recall_summary(d3)
# recall_summary(d5)
# recall_summary(d8)
# recall_summary(d10)
# recall_summary(d12)
# recall_summary(d14)

In [43]:
recall_summary(d3)

Accuracy: 0.7125
Specialist Recall: 0.7267
General Recall: 0.6700


In [44]:
cm_k3 = pd.crosstab(
    d3["expected_service"],
    d3["predicted_label"]
)

cm_k3

predicted_label,abstract_algebra,anatomy,astronomy,general
expected_service,,,,
abstract_algebra,86,0,0,14
anatomy,0,68,2,30
astronomy,1,1,64,34
general,28,2,3,67


In [18]:
error_k3 = d3[ (d3["expected_service"] == "general") & (d3["predicted_label"] == "anatomy")]

In [ ]:
error_k3.

In [24]:
error_k3[
    [
        "query",
        "expected_service",
        "predicted_label",

        "abstract_algebra_zscore",
        "anatomy_zscore",
        "astronomy_zscore",

#         "top_zscore",
#         "second_top_zscore",
#         "zscore_margin",

        "std_score",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
    ]
]

,query,expected_service,predicted_label,abstract_algebra_zscore,anatomy_zscore,astronomy_zscore,std_score,abstract_algebra_score,anatomy_score,astronomy_score
300,don't call me bill anymore - call me bob,general,anatomy,-0.707107,1.414214,-0.707107,0.3536,-12.4375,-11.68750,-12.4375
307,my capital one visa was taken without permission,general,anatomy,-0.805387,1.409428,-0.604040,0.3104,-12.5625,-11.87500,-12.5000
310,"I am traveling to Germany, Will I be able to use my card there?",general,anatomy,-0.967620,1.376997,-0.409378,0.5598,-13.2500,-11.93750,-12.9375
311,"If I feel someone has my card information, can I get a new card?",general,anatomy,-0.568241,1.405649,-0.837408,0.6966,-12.8125,-11.43750,-13.0000
327,tell me what this song is called,general,anatomy,-0.561951,1.404879,-0.842927,0.2224,-11.1875,-10.75000,-11.2500
332,play the next song please,general,anatomy,-0.579324,1.406930,-0.827606,0.2517,-11.2500,-10.75000,-11.3125
336,what's the number of vacation days i have left,general,anatomy,-0.707107,1.414214,-0.707107,0.1768,-11.2500,-10.87500,-11.2500
337,what's on my list of things to remember,general,anatomy,-0.662481,1.413294,-0.750812,0.7076,-8.2500,-6.78125,-8.3125
340,I'm unable to use contactless payments for purchases.,general,anatomy,-0.801052,1.409851,-0.608799,0.6502,-12.6875,-11.25000,-12.5625
343,i would like to change the name on my first bankcard account,general,anatomy,-0.707107,1.414214,-0.707107,0.2062,-12.2500,-11.81250,-12.2500


In [35]:
import numpy as np
from scipy import stats

stats.zscore([-12.4375,-11.68750,-12.4375])

# a = np.average([-12.4375,-11.68750,-12.4375]) k-1
# a

array([-0.70710678,  1.41421356, -0.70710678])

# K5

In [35]:
recall_summary(d5)

Accuracy: 0.9000
Specialist Recall: 0.9020
General Recall: 0.8900


In [36]:
cm_k5 = pd.crosstab(
    d5["expected_service"],
    d5["predicted_label"]
)
cm_k5

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general
expected_service,,,,,,
abstract_algebra,99,0,0,0,1,0
anatomy,0,87,1,0,0,12
astronomy,1,0,96,0,0,3
business_ethics,0,0,0,87,2,11
college_computer_science,17,0,0,0,82,1
general,0,0,2,6,3,89


In [74]:
# error_k5 = d5[ (d5["expected_service"] == "general") & (d5["predicted_label"] == "anatomy")]

In [28]:
# error_k5[
#     [
#         "query",
#         "expected_service",
#         "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",

# #         "top_zscore",
# #         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

#         "abstract_algebra_score",
#         "anatomy_score",
#         "astronomy_score",
#         "business_ethics_score",
#         "college_computer_science_score",
#     ]
# ]

# K8

In [37]:
recall_summary(d8)

Accuracy: 0.8233
Specialist Recall: 0.8137
General Recall: 0.9000


In [38]:
cm_k8 = pd.crosstab(
    d8["expected_service"],
    d8["predicted_label"]
)

cm_k8

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general,high_school_chemistry,high_school_geography,jurisprudence
expected_service,,,,,,,,,
abstract_algebra,99,0,0,0,1,0,0,0,0
anatomy,0,84,1,0,0,13,2,0,0
astronomy,1,0,94,0,0,4,1,0,0
business_ethics,0,0,0,78,2,14,0,3,3
college_computer_science,17,0,0,0,81,2,0,0,0
general,0,0,0,5,3,90,0,2,0
high_school_chemistry,4,0,3,0,0,5,88,0,0
high_school_geography,2,0,2,2,1,40,0,52,1
jurisprudence,0,0,1,0,0,24,0,0,75


In [39]:
# error_k8 = d8[ (d8["expected_service"] == "general") & (d8["predicted_label"] == "anatomy")]

In [40]:
# error_k8[
#     [
#         "query",
#         "expected_service",
#         "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",
#         "high_school_chemistry_zscore",
#         "high_school_geography_zscore",
#         "jurisprudence_zscore",

# #         "top_zscore",
# #         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

#         "abstract_algebra_score",
#         "anatomy_score",
#         "astronomy_score",
#         "business_ethics_score",
#         "college_computer_science_score",
#         "high_school_chemistry_score",
#         "high_school_geography_score",
#         "jurisprudence_score",
#     ]
# ]

# K10

In [41]:
recall_summary(d10)

Accuracy: 0.7945
Specialist Recall: 0.7810
General Recall: 0.9300


In [48]:
cm_k10 = pd.crosstab(
    d10["expected_service"],
    d10["predicted_label"]
)

cm_k10

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,formal_logic,general,high_school_chemistry,high_school_geography,jurisprudence,world_religions
expected_service,,,,,,,,,,,
abstract_algebra,90,0,0,0,0,1,9,0,0,0,0
anatomy,0,96,0,0,0,0,3,1,0,0,0
astronomy,0,0,95,0,0,0,5,0,0,0,0
business_ethics,0,0,0,65,0,1,30,0,0,3,1
college_computer_science,3,0,0,0,70,2,25,0,0,0,0
formal_logic,0,0,0,0,0,93,7,0,0,0,0
general,0,2,3,0,3,0,84,0,0,7,1
high_school_chemistry,0,1,0,0,0,0,8,91,0,0,0
high_school_geography,1,0,1,3,0,1,40,0,41,4,9


In [49]:
# error_k10 = d10[ (d10["expected_service"] == "general") & (d10["predicted_label"] == "high_school_chemistry")]

In [43]:
# error_k10[
#     [
#         "query",
#         "expected_service",
#         "predicted_label",

#         "abstract_algebra_zscore",
#         "anatomy_zscore",
#         "astronomy_zscore",
#         "business_ethics_zscore",
#         "college_computer_science_zscore",
#         "formal_logic_zscore",
#         "high_school_chemistry_zscore",
#         "high_school_geography_zscore",
#         "jurisprudence_zscore",
#         "world_religions_zscore",

#         "top_zscore",
#         "second_top_zscore",
#         "zscore_margin",

#         "std_score",

#         "abstract_algebra_score",
#         "anatomy_score",
#         "astronomy_score",
#         "business_ethics_score",
#         "college_computer_science_score",
#         "formal_logic_score",
#         "high_school_chemistry_score",
#         "high_school_geography_score",
#         "jurisprudence_score",
#         "world_religions_score",
#     ]
# ]